#### Initialize the libraries, the environment and the client

In [1]:
import pandas as pd
import os

try:
    from epo.tipdata.ops import OPSClient, models, exceptions
except ImportError:
    os.system('pip install epo-tipdata-ops')
    from epo.tipdata.ops import OPSClient, models, exceptions

%load_ext dotenv
%dotenv
%load_ext autoreload
%autoreload 2
    
client = OPSClient(
    key=os.getenv("OPS_KEY"), secret=os.getenv("OPS_SECRET")
)

### 7.2 Family Method
#### 7.2.1	family(reference_type, input, constituents, output_type)

* Retrieves the extended patent family information for a given patent, providing details of related patents across different jurisdictions.
* Parameters:

```reference_type (str)```: Type of patent reference (e.g., "publication", "application", or "priority")<br>
```input (Epodoc or Docdb)```: Patent document identifier<br>
```constituents (list[str], optional)```: Additional information to retrieve (bibliographic or legal data)<br>
```output_type (str)```: Format of the returned data (raw XML or Pandas DataFrame)<br>

#### 7.2.2	Example of fetching from Docdb a family by publication
This example retrieves the INPADOC family information (patent numbers of related applications) for a published Japanese patent identified by its document number stored in the Docdb object.

In [2]:
family_xml = client.family(
    reference_type="publication",    
    input=models.Docdb("4106326", "EP", "A1"), # input=models.Docdb("2101496", "EP", "B1"),  # input=models.Docdb("2995292", "JP", "B1"),
    constituents=["biblio"],
    output_type="raw",
)
print(family_xml[:2000])

family_df = client.family(
    reference_type="publication",    
    input=models.Docdb("4106326", "EP", "A1"), # input=models.Docdb("2101496", "EP", "B1"),  # input=models.Docdb("2995292", "JP", "B1"),
    constituents=None,
    output_type="dataframe",
)
family_df

<?xml version="1.0" encoding="UTF-8"?><?xml-stylesheet type="text/xsl" href="../../../../../style/family-biblio.xsl"?>
<ops:world-patent-data xmlns="http://www.epo.org/exchange" xmlns:ops="http://ops.epo.org" xmlns:xlink="http://www.w3.org/1999/xlink">
    <ops:patent-family legal="false" total-result-count="2">
        <ops:publication-reference>
            <document-id document-id-type="docdb">
                <country>EP</country>
                <doc-number>4106326</doc-number>
                <kind>A1</kind>
            </document-id>
        </ops:publication-reference>
        <ops:family-member family-id="82067485">
            <publication-reference>
                <document-id document-id-type="docdb">
                    <country>EP</country>
                    <doc-number>4106326</doc-number>
                    <kind>A1</kind>
                    <date>20221221</date>
                </document-id>
                <document-id document-id-type="epodoc">
                

,legal,total-result-count,ops:family-member|family-id,ops:family-member|publication-reference,ops:family-member|application-reference,ops:family-member|priority-claim,ops:publication-reference|document-id|document-id-type,ops:publication-reference|document-id|country,ops:publication-reference|document-id|doc-number,ops:publication-reference|document-id|kind
0,false,2,82067485,{'document-id': [{'@document-id-type': 'docdb'...,"{'@doc-id': '574299984', '@is-representative':...","[{'@sequence': '1', '@kind': 'national', 'docu...",docdb,EP,4106326,A1
1,false,2,82067485,{'document-id': [{'@document-id-type': 'docdb'...,"{'@doc-id': '584535325', 'document-id': {'@doc...","[{'@sequence': '1', '@kind': 'national', 'docu...",docdb,EP,4106326,A1


#### 7.2.2.1 The variable family_xml will be used for programmatic purposes to:
* Parse the XML response
* Define namespaces correctly

In [3]:
import xml.etree.ElementTree as ET
from pprint import pprint

# Parse the XML data
root = ET.fromstring(family_xml)
pprint(family_xml[:1000])

# Define namespaces
namespaces = {
    'ops': 'http://ops.epo.org',
    '': 'http://www.epo.org/exchange',
    'xlink': 'http://www.w3.org/1999/xlink'
}

('<?xml version="1.0" encoding="UTF-8"?><?xml-stylesheet type="text/xsl" '
 'href="../../../../../style/family-biblio.xsl"?>\n'
 '<ops:world-patent-data xmlns="http://www.epo.org/exchange" '
 'xmlns:ops="http://ops.epo.org" xmlns:xlink="http://www.w3.org/1999/xlink">\n'
 '    <ops:patent-family legal="false" total-result-count="2">\n'
 '        <ops:publication-reference>\n'
 '            <document-id document-id-type="docdb">\n'
 '                <country>EP</country>\n'
 '                <doc-number>4106326</doc-number>\n'
 '                <kind>A1</kind>\n'
 '            </document-id>\n'
 '        </ops:publication-reference>\n'
 '        <ops:family-member family-id="82067485">\n'
 '            <publication-reference>\n'
 '                <document-id document-id-type="docdb">\n'
 '                    <country>EP</country>\n'
 '                    <doc-number>4106326</doc-number>\n'
 '                    <kind>A1</kind>\n'
 '                    <date>20221221</date>\n'
 '        

This code works similarly to the prior example for published_data. Here is what it does:

* We import the Docdb class to represent the patent document.
* We define the document number for the published patent.
* We create an Docdb object with that number.
* The family method is called with the following arguments:
```reference_type```: Set to "publication" as we're looking for a published patent.
```input```: The input object containing the document number.
```constituents```: Set to None by default, retrieving all family information. (Optional: Set to a list containing "biblio" and/or "legal" for specific data).
```output_type```: Set to "dataframe" (optional) to get the data as a Pandas DataFrame for easier manipulation.

The family method retrieves the INPADOC family information for the input patent. It then uses the _get_content method to convert the response to the desired format (DataFrame in this case, or raw XML if not specified). The last line checks the type of the returned data. If it's a DataFrame, it prints the first few rows containing information about the family members (patent numbers etc.). If it's raw XML, it informs the user that additional parsing might be needed to access the data. In this case the command pprint(family) facilitates reading of XML data.

#### 7.2.2.2 We extract the document details:
* the document number
* the country code
* the kind code

In [4]:
# Initialize the set to keep track of unique document numbers
unique_documents = set()
# Extract all document numbers
document_details = []

# Find all 'document-id' elements
for doc_id in root.findall(".//{http://www.epo.org/exchange}document-id"):
    # Debug print to show the document_id element
    # print(f"Found document-id element: {ET.tostring(doc_id, encoding='unicode')}")    
    doc_number = doc_id.find('{http://www.epo.org/exchange}doc-number')
    country = doc_id.find("{http://www.epo.org/exchange}country")
    kind = doc_id.find("{http://www.epo.org/exchange}kind")    
    if doc_number is not None and doc_number.text not in unique_documents and country is not None and kind is not None:
        # Add the document number to the set
        unique_documents.add(doc_number.text)
        document_details.append({
            'doc_number': doc_number.text,
            'country': country.text,
            'kind': kind.text
        })
        
# Print the extracted document numbers
print("document details:", document_details[:50])

document details: [{'doc_number': '4106326', 'country': 'EP', 'kind': 'A1'}, {'doc_number': '22179301', 'country': 'EP', 'kind': 'A'}, {'doc_number': '202217806216', 'country': 'US', 'kind': 'A'}, {'doc_number': '202163202527', 'country': 'US', 'kind': 'P'}, {'doc_number': '202762632025', 'country': 'US', 'kind': None}, {'doc_number': '8395653', 'country': 'US', 'kind': 'B2'}, {'doc_number': '8842161', 'country': 'US', 'kind': 'B2'}, {'doc_number': '9030520', 'country': 'US', 'kind': 'B2'}, {'doc_number': '9542603', 'country': 'US', 'kind': 'B2'}, {'doc_number': '9723260', 'country': 'US', 'kind': 'B2'}, {'doc_number': '9800835', 'country': 'US', 'kind': 'B2'}, {'doc_number': '10091412', 'country': 'US', 'kind': 'B1'}, {'doc_number': '10122972', 'country': 'US', 'kind': 'B2'}, {'doc_number': '10187579', 'country': 'US', 'kind': 'B1'}, {'doc_number': '10574899', 'country': 'US', 'kind': 'B2'}, {'doc_number': '2020103078', 'country': 'WO', 'kind': 'A1'}, {'doc_number': '2020103068', 'cou

In [5]:
family = client.family(
    reference_type="application",    
    input=models.Docdb("22179301", "EP", "A"),  # input=models.Docdb("09164213", "EP", "A"), # input=models.Docdb("2101496", "EP", "B1"), # input=models.Docdb("2792747", "EP", "A1"), 
    constituents=["biblio", "legal"],
    output_type="dataframe",
)

# Print the columns to see if document details are present
print(family.columns)

# Check if document details are present in the DataFrame
if all(col in family.columns for col in ['ops:publication-reference|document-id|doc-number', 
                                         'ops:publication-reference|document-id|country', 
                                         'ops:publication-reference|document-id|kind']):
    # Extract relevant document details and remove duplicates
    document_details = family[['ops:publication-reference|document-id|doc-number', 
                               'ops:publication-reference|document-id|country', 
                               'ops:publication-reference|document-id|kind']].drop_duplicates()

    # Rename columns for easier readability
    document_details.columns = ['Document Number', 'Country', 'Kind']
    
    # Print document details
    print(document_details)
else:
    print("Document details not found in DataFrame")

# Reset to default options if needed with pd.reset_option('display.max_colwidth')
pd.set_option('display.max_colwidth', 2000)

family

Index(['legal', 'total-result-count', 'ops:family-member|family-id',
       'ops:family-member|publication-reference',
       'ops:family-member|application-reference',
       'ops:family-member|priority-claim', 'ops:family-member|ops:legal',
       'ops:family-member|exchange-document',
       'ops:application-reference|document-id|document-id-type',
       'ops:application-reference|document-id|country',
       'ops:application-reference|document-id|doc-number',
       'ops:application-reference|document-id|kind'],
      dtype='object')
Document details not found in DataFrame


,legal,total-result-count,ops:family-member|family-id,ops:family-member|publication-reference,ops:family-member|application-reference,ops:family-member|priority-claim,ops:family-member|ops:legal,ops:family-member|exchange-document,ops:application-reference|document-id|document-id-type,ops:application-reference|document-id|country,ops:application-reference|document-id|doc-number,ops:application-reference|document-id|kind
0,true,2,82067485,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '4106326', 'kind': 'A1', 'date': '20221221'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP4106326', 'date': '20221221'}]}","{'@doc-id': '574299984', '@is-representative': 'YES', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '22179301', 'kind': 'A', 'date': '20220615'}}","[{'@sequence': '1', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202217806216', 'kind': 'A', 'date': '20220609'}, 'priority-active-indicator': 'YES'}, {'@sequence': '2', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202163202527', 'kind': 'P', 'date': '20210615'}, 'priority-active-indicator': 'YES'}]","[{'@code': 'AK ', '@desc': 'DESIGNATED CONTRACTING STATES', '@infl': '+', '@dateMigr': '00010101', 'ops:pre': [{'@line': '00001', '#text': 'EP 22179301A 2022-12-21AK +DESIGNATED CONTRACTING STATES Kind Code of Ref Document A1'}, {'@line': '00002', '#text': 'EP 22179301A 2022-12-21AK +DESIGNATED CONTRACTING STATES Designated State(s) AL AT BE BG CH CY CZ DE DK EE ES FI FR GB GR HR HU IE IS IT LI LT LU LV MC MK MT NL NO PL PT RO RS SE SI SK SM TR'}], 'ops:L001EP': {'@desc': 'Country Code', '#text': 'EP'}, 'ops:L002EP': {'@desc': 'Filing / Published Document', '#text': 'F'}, 'ops:L003EP': {'@desc': 'Document Number', '#text': '22179301'}, 'ops:L004EP': {'@desc': 'Kind Code', '#text': 'A'}, 'ops:L005EP': {'@desc': 'IPR Type', '#text': 'PI'}, 'ops:L007EP': {'@desc': 'Gazette DATE', '#text': '2022-12-21'}, 'ops:L008EP': {'@desc': 'Legal Event Code 1', '#text': 'AK'}, 'ops:L018EP': {'@desc': 'DATE last exchanged', '#text': '2022-12-24'}, 'ops:L019EP': {'@desc': 'DATE first created', '#text': '2022-12-21'}, 'ops:L500EP': {'ops:L506EP': {'@desc': 'Kind Code of Ref Document', '#text': 'A1'}, 'ops:L507EP': {'@desc': 'Designated State(s)', '#text': 'AL AT BE BG CH CY CZ DE DK EE ES FI FR GB GR HR HU IE IS IT LI LT LU LV MC MK MT NL NO PL PT RO RS SE SI SK SM TR'}}}, {'@code': 'PUAI', '@desc': 'PUBLIC REFERENCE MADE UNDER ARTICLE 153(3) EPC TO A PUBLISHED INTERNATIONAL APPLICATION THAT HAS ENTERED THE EUROPEAN PHASE', '@infl': ' ', '@dateMigr': '00010101', 'ops:pre': {'@line': '00001', '#text': 'EP 22179301A 2022-11-18PUAI PUBLIC REFERENCE MADE UNDER ARTICLE 153(3) EPC TO A PUBLISHED INTERNATIONAL APPLICATION THAT HAS ENTERED THE EUROPEAN PHASE Free Format Text ORIGINAL CODE: 0009012'}, 'ops:L001EP': {'@desc': 'Country Code', '#text': 'EP'}, 'ops:L002EP': {'@desc': 'Filing / Published Document', '#text': 'F'}, 'ops:L003EP': {'@desc': 'Document Number', '#text': '22179301'}, 'ops:L004EP': {'@desc': 'Kind Code', '#text': 'A'}, 'ops:L005EP': {'@desc': 'IPR Type',...","{'@system': 'ops.epo.org', '@family-id': '82067485', '@country': 'EP', '@doc-number': '4106326', '@kind': 'A1', 'bibliographic-data': {'publication-reference': {'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '4106326', 'kind': 'A1', 'date': '20221221'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP4106326', 'date': '20221221'}]}, 'classifications-ipcr': {'classification-ipcr': {'@sequence': '1', 'text': 'H04N 7/ 14 A I'}}, 'patent-classifications': {'patent-classification': [{'@sequence': '1', 'classification-scheme': {'@office': 'EP', '@scheme': 'CPCI'}, 'section': 'H', 'class': '04', 'subclass': 'N', 'main-group': '7', 'subgroup': '147', 'classification-value': 'I', 'generating-office': 'US'}, {'@sequence': '2', 'classificati

#### 7.2.3	Example of fetching bibliographic data and legal data of a family 
* by retrieving the bibliographic data of the application
* and by retrieving the legal data of the publication
* before combining them

#### 7.2.3.1 The bibliographic data and the legal data are derived from the family of ```"22179301", "EP", "A"``` and ```"4106326", "EP", "A1"``` as stored in Docdb.

In [6]:
import pandas as pd

# Reset to default options if needed with pd.reset_option('display.max_colwidth')
pd.set_option('display.max_colwidth', 2000)

# Fetch bibliographic data for published patent application
family1 = client.family(
    reference_type="application",  # publication, application, priority
    input=models.Docdb("22179301", "EP", "A"),  # original, docdb, epodoc
    constituents=None,
    output_type="Dataframe"  # optional, xml or DataFrame format
)
display(family1)

family2 = client.family(
    reference_type="publication",  # publication, application, priority
    input=models.Docdb("4106326", "EP", "A1"),  # original, docdb, epodoc
    constituents=["legal"],
    output_type="Dataframe"  # optional, xml or DataFrame format
)
display(family2)

# # Inspect the columns
print("Family1 Columns:", family1.columns)
print("Family2 Columns:", family2.columns)

# Print the number of rows in each DataFrame
print(f"Number of rows in family1: {len(family1)}")
print(f"Number of rows in family2: {len(family2)}")

,legal,total-result-count,ops:family-member|family-id,ops:family-member|publication-reference,ops:family-member|application-reference,ops:family-member|priority-claim,ops:application-reference|document-id|document-id-type,ops:application-reference|document-id|country,ops:application-reference|document-id|doc-number,ops:application-reference|document-id|kind
0,false,2,82067485,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '4106326', 'kind': 'A1', 'date': '20221221'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP4106326', 'date': '20221221'}]}","{'@doc-id': '574299984', '@is-representative': 'YES', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '22179301', 'kind': 'A', 'date': '20220615'}}","[{'@sequence': '1', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202217806216', 'kind': 'A', 'date': '20220609'}, 'priority-active-indicator': 'YES'}, {'@sequence': '2', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202163202527', 'kind': 'P', 'date': '20210615'}, 'priority-active-indicator': 'YES'}]",docdb,EP,22179301,A
1,false,2,82067485,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '2022400244', 'kind': 'A1', 'date': '20221215'}, {'@document-id-type': 'epodoc', 'doc-number': 'US2022400244', 'date': '20221215'}]}","{'@doc-id': '584535325', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202217806216', 'kind': 'A', 'date': '20220609'}}","[{'@sequence': '1', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202217806216', 'kind': 'A', 'date': '20220609'}, 'priority-active-indicator': 'YES'}, {'@sequence': '2', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202163202527', 'kind': 'P', 'date': '20210615'}, 'priority-active-indicator': 'YES'}]",docdb,EP,22179301,A


,legal,total-result-count,ops:family-member|family-id,ops:family-member|publication-reference,ops:family-member|application-reference,ops:family-member|priority-claim,ops:family-member|ops:legal,ops:publication-reference|document-id|document-id-type,ops:publication-reference|document-id|country,ops:publication-reference|document-id|doc-number,ops:publication-reference|document-id|kind
0,true,2,82067485,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '4106326', 'kind': 'A1', 'date': '20221221'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP4106326', 'date': '20221221'}]}","{'@doc-id': '574299984', '@is-representative': 'YES', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '22179301', 'kind': 'A', 'date': '20220615'}}","[{'@sequence': '1', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202217806216', 'kind': 'A', 'date': '20220609'}, 'priority-active-indicator': 'YES'}, {'@sequence': '2', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202163202527', 'kind': 'P', 'date': '20210615'}, 'priority-active-indicator': 'YES'}]","[{'@code': 'AK ', '@desc': 'DESIGNATED CONTRACTING STATES', '@infl': '+', '@dateMigr': '00010101', 'ops:pre': [{'@line': '00001', '#text': 'EP 22179301A 2022-12-21AK +DESIGNATED CONTRACTING STATES Kind Code of Ref Document A1'}, {'@line': '00002', '#text': 'EP 22179301A 2022-12-21AK +DESIGNATED CONTRACTING STATES Designated State(s) AL AT BE BG CH CY CZ DE DK EE ES FI FR GB GR HR HU IE IS IT LI LT LU LV MC MK MT NL NO PL PT RO RS SE SI SK SM TR'}], 'ops:L001EP': {'@desc': 'Country Code', '#text': 'EP'}, 'ops:L002EP': {'@desc': 'Filing / Published Document', '#text': 'F'}, 'ops:L003EP': {'@desc': 'Document Number', '#text': '22179301'}, 'ops:L004EP': {'@desc': 'Kind Code', '#text': 'A'}, 'ops:L005EP': {'@desc': 'IPR Type', '#text': 'PI'}, 'ops:L007EP': {'@desc': 'Gazette DATE', '#text': '2022-12-21'}, 'ops:L008EP': {'@desc': 'Legal Event Code 1', '#text': 'AK'}, 'ops:L018EP': {'@desc': 'DATE last exchanged', '#text': '2022-12-24'}, 'ops:L019EP': {'@desc': 'DATE first created', '#text': '2022-12-21'}, 'ops:L500EP': {'ops:L506EP': {'@desc': 'Kind Code of Ref Document', '#text': 'A1'}, 'ops:L507EP': {'@desc': 'Designated State(s)', '#text': 'AL AT BE BG CH CY CZ DE DK EE ES FI FR GB GR HR HU IE IS IT LI LT LU LV MC MK MT NL NO PL PT RO RS SE SI SK SM TR'}}}, {'@code': 'PUAI', '@desc': 'PUBLIC REFERENCE MADE UNDER ARTICLE 153(3) EPC TO A PUBLISHED INTERNATIONAL APPLICATION THAT HAS ENTERED THE EUROPEAN PHASE', '@infl': ' ', '@dateMigr': '00010101', 'ops:pre': {'@line': '00001', '#text': 'EP 22179301A 2022-11-18PUAI PUBLIC REFERENCE MADE UNDER ARTICLE 153(3) EPC TO A PUBLISHED INTERNATIONAL APPLICATION THAT HAS ENTERED THE EUROPEAN PHASE Free Format Text ORIGINAL CODE: 0009012'}, 'ops:L001EP': {'@desc': 'Country Code', '#text': 'EP'}, 'ops:L002EP': {'@desc': 'Filing / Published Document', '#text': 'F'}, 'ops:L003EP': {'@desc': 'Document Number', '#text': '22179301'}, 'ops:L004EP': {'@desc': 'Kind Code', '#text': 'A'}, 'ops:L005EP': {'@desc': 'IPR Type',...",docdb,EP,4106326,A1
1,true,2,82067485,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '2022400244', 'kind': 'A1', 'date': '20221215'}, {'@document-id-type': 'epodoc', 'doc-number': 'US2022400244', 'date': '20221215'}]}","{'@doc-id': '584535325', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202217806216', 'kind': 'A', 'date': '20220609'}}","[{'@sequence': '1', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202217806216', 'kind': 'A', 'date': '20220609'}, 'priority-active-indicator': 'YES'}, {'@sequence': '2', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202163202527', 'kind': 'P', 'date': '20210615'}, 'priority-active-indicator': '

Family1 Columns: Index(['legal', 'total-result-count', 'ops:family-member|family-id',
       'ops:family-member|publication-reference',
       'ops:family-member|application-reference',
       'ops:family-member|priority-claim',
       'ops:application-reference|document-id|document-id-type',
       'ops:application-reference|document-id|country',
       'ops:application-reference|document-id|doc-number',
       'ops:application-reference|document-id|kind'],
      dtype='object')
Family2 Columns: Index(['legal', 'total-result-count', 'ops:family-member|family-id',
       'ops:family-member|publication-reference',
       'ops:family-member|application-reference',
       'ops:family-member|priority-claim', 'ops:family-member|ops:legal',
       'ops:publication-reference|document-id|document-id-type',
       'ops:publication-reference|document-id|country',
       'ops:publication-reference|document-id|doc-number',
       'ops:publication-reference|document-id|kind'],
      dtype='object')

#### 7.2.3.2 Combining Family1 and Family2 dataframes 
* defining the common column to Family1 and Family2 dataframes: 'ops:family-member|family-id'
* Perform an outer join to diagnose missing keys
* Print the combined DataFrame with the outer join
* Show how many rows are in each category (both, left_only, right_only)
* Display rows with mismatched keys
* Perform the actual merge using inner join
* Display the combined DataFrame

In [7]:
# Assuming the common column to Family1 and Family2 dataframes is 'ops:family-member|family-id'
common_column = 'ops:family-member|family-id'

# Perform an outer join to diagnose missing keys
combined_family_outer = pd.merge(family1, family2, on=common_column, how='outer', indicator=True)

# Print the combined DataFrame with the outer join
print(combined_family_outer[:5])

# Show how many rows are in each category (both, left_only, right_only)
print(combined_family_outer['_merge'].value_counts()[:5])

# Display rows with mismatched keys
print(combined_family_outer[combined_family_outer['_merge'] != 'both'])

# Perform the actual merge using inner join
combined_family = pd.merge(family1, family2, on=common_column, how='inner')

# Display the combined DataFrame
combined_family

  legal_x total-result-count_x ops:family-member|family-id  \
0   false                    2                    82067485   
1   false                    2                    82067485   
2   false                    2                    82067485   
3   false                    2                    82067485   

                                                                                                                                                                             ops:family-member|publication-reference_x  \
0        {'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '4106326', 'kind': 'A1', 'date': '20221221'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP4106326', 'date': '20221221'}]}   
1        {'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '4106326', 'kind': 'A1', 'date': '20221221'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP4106326', 'date': '20221221'}]}   
2  {'document-id': [{'@document-id-

,legal_x,total-result-count_x,ops:family-member|family-id,ops:family-member|publication-reference_x,ops:family-member|application-reference_x,ops:family-member|priority-claim_x,ops:application-reference|document-id|document-id-type,ops:application-reference|document-id|country,ops:application-reference|document-id|doc-number,ops:application-reference|document-id|kind,legal_y,total-result-count_y,ops:family-member|publication-reference_y,ops:family-member|application-reference_y,ops:family-member|priority-claim_y,ops:family-member|ops:legal,ops:publication-reference|document-id|document-id-type,ops:publication-reference|document-id|country,ops:publication-reference|document-id|doc-number,ops:publication-reference|document-id|kind
0,false,2,82067485,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '4106326', 'kind': 'A1', 'date': '20221221'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP4106326', 'date': '20221221'}]}","{'@doc-id': '574299984', '@is-representative': 'YES', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '22179301', 'kind': 'A', 'date': '20220615'}}","[{'@sequence': '1', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202217806216', 'kind': 'A', 'date': '20220609'}, 'priority-active-indicator': 'YES'}, {'@sequence': '2', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202163202527', 'kind': 'P', 'date': '20210615'}, 'priority-active-indicator': 'YES'}]",docdb,EP,22179301,A,true,2,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '4106326', 'kind': 'A1', 'date': '20221221'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP4106326', 'date': '20221221'}]}","{'@doc-id': '574299984', '@is-representative': 'YES', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '22179301', 'kind': 'A', 'date': '20220615'}}","[{'@sequence': '1', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202217806216', 'kind': 'A', 'date': '20220609'}, 'priority-active-indicator': 'YES'}, {'@sequence': '2', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'US', 'doc-number': '202163202527', 'kind': 'P', 'date': '20210615'}, 'priority-active-indicator': 'YES'}]","[{'@code': 'AK ', '@desc': 'DESIGNATED CONTRACTING STATES', '@infl': '+', '@dateMigr': '00010101', 'ops:pre': [{'@line': '00001', '#text': 'EP 22179301A 2022-12-21AK +DESIGNATED CONTRACTING STATES Kind Code of Ref Document A1'}, {'@line': '00002', '#text': 'EP 22179301A 2022-12-21AK +DESIGNATED CONTRACTING STATES Designated State(s) AL AT BE BG CH CY CZ DE DK EE ES FI FR GB GR HR HU IE IS IT LI LT LU LV MC MK MT NL NO PL PT RO RS SE SI SK SM TR'}], 'ops:L001EP': {'@desc': 'Country Code', '#text': 'EP'}, 'ops:L002EP': {'@desc': 'Filing / Published Document', '#text': 'F'}, 'ops:L003EP': {'@desc': 'Document Number', '#text': '22179301'}, 'ops:L004EP': {'@desc': 'Kind Code', '#text': 'A'}, 'ops:L005EP': {'@desc': 'IPR Type', '#text': 'PI'}, 'ops:L007EP': {'@desc': 'Gazette DATE', '#text': '2022-12-21'}, 'ops:L008EP': {'@desc': 'Legal Event Code 1', '#text': 'AK'}, 'ops:L018EP': {'@desc': 'DATE last exchanged', '#text': '2022-12-24'}, 'ops:L019EP': {'@desc': 'DATE first created', '#text': '2022-12-21'}, 'ops:L500EP': {'ops:L506EP': {'@desc': 'Kind Code of Ref Document', '#text': 'A1'}, 'ops:L507EP': {'@desc': 'Designated State(s)', '#text': 'AL AT BE BG CH CY CZ DE DK EE ES FI FR GB GR HR HU IE IS IT LI LT LU LV MC MK MT NL NO PL PT RO RS SE SI SK SM TR'}}}, {'@code': 'PUAI', '@desc': 'PUBLIC REFERENCE MADE UNDER ARTICLE 153(3) EPC TO A PUBLISHED INTERNATIONAL APPLICATION THAT HAS ENTERED THE EUROPEAN PHASE', '@infl': ' ', '@dateMigr': '00010101', 'ops:pre': {'@line': '00001', '#text': 'EP 22179301A 2022-11-18PUAI PUBLIC REFERENCE MADE UNDER ARTICLE 153(3) EPC TO A PUBLISHED INTERNATIONAL APPLIC

#### 7.2.3.3 If ```ops:family-member|ops:legal``` contains JSON-like data structures, you might iterate through the rows to access specific fields within each legal detail:
* Define the function to process and combine data to a DataFrame named combined_df
* This example assumes ```ops:family-member|ops:legal``` contains a list of dictionaries with @code and @desc attributes. 

In [8]:
import pandas as pd

# Function to process and combine data from the DataFrame
def process_family_data(family_df, num_rows=5):
    combined_data = []
    
    for index, row in family_df.head(num_rows).iterrows():
        # Initialize a dictionary to store combined data for the current row
        row_data = {
            'Family ID': row.get('ops:family-member|family-id', 'Unknown'),
            'Application Reference': row.get('ops:family-member|application-reference', 'Unknown'),
            'Priority Claim': row.get('ops:family-member|priority-claim', 'Unknown'),
        }
        
        # Extract and process nested document-id data
        pub_reference = row.get('ops:family-member|publication-reference', {})
        
        if isinstance(pub_reference, dict) and 'document-id' in pub_reference:
            document_ids = pub_reference['document-id']            
            if isinstance(document_ids, list):
                for doc_id in document_ids:
                    combined_row = row_data.copy()  # Copy base row_data to avoid modifying it
                    combined_row.update({
                        'Document ID Type': doc_id.get('@document-id-type', 'Unknown'),
                        'Country': doc_id.get('country', 'Unknown'),
                        'Document Number': doc_id.get('doc-number', 'Unknown'),
                        'Kind': doc_id.get('kind', 'Unknown'),
                    })
                    combined_data.append(combined_row)  # Append the combined row data to the list
            else:
                combined_data.append(row_data)  # Append if no document-id list
        else:
            combined_data.append(row_data)  # Append if no document-id data
    
    # Extract and add legal information
    for item in family_df['ops:family-member|ops:legal']:
        if isinstance(item, list):
            for legal_item in item:
                legal_code = legal_item.get('@code', 'Unknown')
                legal_description = legal_item.get('@desc', 'Unknown')
                # Add legal information to combined data
                for data in combined_data:
                    data.update({
                        'Legal Code': legal_code,
                        'Legal Description': legal_description
                    })
    
    # Convert the combined data into a DataFrame
    combined_df = pd.DataFrame(combined_data)
    
    return combined_df

#### 7.2.3.4 Print the columns of the DataFrame to verify the presence of the target column

In [9]:
# Print the columns of the DataFrame to verify the presence of the target column
print("DataFrame Columns:", family.columns)

# Example usage
combined_df = process_family_data(family)
print(combined_df)

DataFrame Columns: Index(['legal', 'total-result-count', 'ops:family-member|family-id',
       'ops:family-member|publication-reference',
       'ops:family-member|application-reference',
       'ops:family-member|priority-claim', 'ops:family-member|ops:legal',
       'ops:family-member|exchange-document',
       'ops:application-reference|document-id|document-id-type',
       'ops:application-reference|document-id|country',
       'ops:application-reference|document-id|doc-number',
       'ops:application-reference|document-id|kind'],
      dtype='object')
  Family ID  \
0  82067485   
1  82067485   
2  82067485   
3  82067485   

                                                                                                                                                              Application Reference  \
0  {'@doc-id': '574299984', '@is-representative': 'YES', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '22179301', 'kind': 'A', 'date': '20220615'

In [10]:
import pandas as pd
from collections import Counter

# Fetch bibliographic data for published patent application
family_data = client.family(
    reference_type="application",  # publication, application, priority
    input=models.Docdb("09164213", "EP", "A"),  # original, docdb, epodoc
    constituents=None,
    output_type="Dataframe"  # optional, xml or DataFrame format
)
# display(family_data)

# Check if data is retrieved
if family_data.empty:
    print("No family data found for this publication number.")
else:
    # Extract relevant columns
    family_df = family_data[[        
        "ops:family-member|family-id",
        "ops:family-member|publication-reference",
        "ops:family-member|priority-claim"
    ]]

    # Rename columns for readability
    family_df.columns = [
        "Family ID",
        "Publication Reference",
        "Priority Claim"
    ]

    # Extract country codes from 'Publication Reference'
    country_codes = []
    for pub_ref in family_df["Publication Reference"]:
        if isinstance(pub_ref, dict) and "document-id" in pub_ref:
            for doc in pub_ref["document-id"]:
                if "@document-id-type" in doc and doc["@document-id-type"] == "docdb":
                    country_codes.append(doc.get("country", "Unknown"))

    # Generate statistics on country occurrences
    country_counts = Counter(country_codes)
    country_stats_df = pd.DataFrame(country_counts.items(), columns=["Country", "Count"]).sort_values(by="Count", ascending=False)

    # Add total sum row
    total_count = country_stats_df["Count"].sum()
    total_row = pd.DataFrame([["Total", total_count]], columns=["Country", "Count"])
    country_stats_df = pd.concat([country_stats_df, total_row], ignore_index=True)
    
    # Display the structured DataFrame and statistics
    display(family_df)
    print("\nFamily Member Country Statistics:")
    display(country_stats_df)
    


,Family ID,Publication Reference,Priority Claim
0,27290872,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '2101496', 'kind': 'A2', 'date': '20090916'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP2101496', 'date': '20090916'}]}","[{'@sequence': '1', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'JP', 'doc-number': '32377096', 'kind': 'A', 'date': '19961204'}, 'priority-active-indicator': 'YES'}, {'@sequence': '2', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'JP', 'doc-number': '34728496', 'kind': 'A', 'date': '19961226'}, 'priority-active-indicator': 'YES'}, {'@sequence': '3', '@kind': 'regional', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '06011611', 'kind': 'A', 'date': '19970228'}, 'priority-active-indicator': 'NO', 'priority-linkage-type': '3'}, {'@sequence': '4', '@kind': 'regional', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '97903634', 'kind': 'A', 'date': '19970228'}, 'priority-active-indicator': 'NO', 'priority-linkage-type': '3'}, {'@sequence': '5', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'JP', 'doc-number': '4158396', 'kind': 'A', 'date': '19960228'}, 'priority-active-indicator': 'YES'}]"
1,27290872,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '2101496', 'kind': 'A3', 'date': '20091104'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP2101496', 'date': '20091104'}]}","[{'@sequence': '1', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'JP', 'doc-number': '32377096', 'kind': 'A', 'date': '19961204'}, 'priority-active-indicator': 'YES'}, {'@sequence': '2', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'JP', 'doc-number': '34728496', 'kind': 'A', 'date': '19961226'}, 'priority-active-indicator': 'YES'}, {'@sequence': '3', '@kind': 'regional', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '06011611', 'kind': 'A', 'date': '19970228'}, 'priority-active-indicator': 'NO', 'priority-linkage-type': '3'}, {'@sequence': '4', '@kind': 'regional', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '97903634', 'kind': 'A', 'date': '19970228'}, 'priority-active-indicator': 'NO', 'priority-linkage-type': '3'}, {'@sequence': '5', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'JP', 'doc-number': '4158396', 'kind': 'A', 'date': '19960228'}, 'priority-active-indicator': 'YES'}]"
2,27290872,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '2101496', 'kind': 'B1', 'date': '20130123'}, {'@document-id-type': 'epodoc', 'doc-number': 'EP2101496', 'date': '20130123'}]}","[{'@sequence': '1', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'JP', 'doc-number': '32377096', 'kind': 'A', 'date': '19961204'}, 'priority-active-indicator': 'YES'}, {'@sequence': '2', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'JP', 'doc-number': '34728496', 'kind': 'A', 'date': '19961226'}, 'priority-active-indicator': 'YES'}, {'@sequence': '3', '@kind': 'regional', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '06011611', 'kind': 'A', 'date': '19970228'}, 'priority-active-indicator': 'NO', 'priority-linkage-type': '3'}, {'@sequence': '4', '@kind': 'regional', 'document-id': {'@document-id-type': 'docdb', 'country': 'EP', 'doc-number': '97903634', 'kind': 'A', 'date': '19970228'}, 'priority-active-indicator': 'NO', 'priority-linkage-type': '3'}, {'@sequence': '5', '@kind': 'national', 'document-id': {'@document-id-type': 'docdb', 'country': 'JP', 'doc-number': '4158396', 'kind': 'A', 'date': '19960228'}, 'priority-active-indicator': 'YES'}]"
3,27477573,"{'document-id': [{'@document-id-type': 'docdb', 'country': 'CA', 'doc-number': '2273891', 'kind': 'A1',


Family Member Country Statistics:


,Country,Count
0,JP,135
1,US,82
2,EP,74
3,CN,20
4,DE,10
5,KR,8
6,HK,6
7,CA,4
8,WO,4
9,SG,2
